# Advanced CVE Prioritization with Graph Neural Networks

**Date:** January 27, 2026  
**Phase:** 7 - Advanced Models  
**Objective:** Implement and evaluate graph-based models for CVE prioritization

---

## Overview

This notebook explores advanced machine learning models for CVE prioritization:

1. **DiffusionRank**: Random walk-based algorithm on vulnerability similarity graph
2. **RGCN**: Relational Graph Convolutional Network modeling CVE-CWE relationships  
3. **Ensemble**: Meta-learner combining multiple model predictions

### Why Graph Models?

Traditional ML models (LambdaRank, XGBoost) treat CVEs as independent entities. Graph models capture:
- **Structural relationships**: CVE → CWE → Other CVEs
- **Similarity propagation**: High-priority CVEs influence similar vulnerabilities
- **Network effects**: Exploited CVEs in same CWE category are more risky

### Expected Improvements

- **NDCG@20**: 0.95 → 0.97 (+2% lift)
- **Healthcare Recall**: Better identification of medical device CVEs
- **Explainability**: Graph paths show why CVEs are related

In [ ]:
# Core imports
import sys
import sqlite3
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Graph libraries
import networkx as nx
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

# Local imports
sys.path.append('..')
from src.core.database import CVEDatabase
from src.features.engineering import create_all_features
from src.evaluation.metrics import ndcg_at_k, precision_at_k

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Imports successful")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NetworkX: {nx.__version__}")
print(f"LightGBM: {lgb.__version__}")

## 1. Data Loading & Preprocessing

Load CVE data with enrichments and extract graph structure.

In [ ]:
# Connect to database
db_path = Path('../data/cve_database.db')
db = CVEDatabase(str(db_path))

print(f"Database: {db_path}")
print(f"Total CVEs: {db.count_cves():,}")

In [ ]:
# Load CVE data with enrichments
query = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    c.cwe,
    e.epss_score,
    e.epss_percentile,
    e.kev_flag,
    e.is_healthcare,
    e.attack_flag,
    e.attack_technique_count,
    e.chpl_flag,
    e.label
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.published >= '2024-01-01'
    AND c.cvss IS NOT NULL
    AND e.label IS NOT NULL
ORDER BY c.published DESC
"""

df = pd.read_sql(query, db.conn)
df['published'] = pd.to_datetime(df['published'])

print(f"Loaded CVEs: {len(df):,}")
print(f"Date range: {df['published'].min()} to {df['published'].max()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

df.head()

In [ ]:
# Extract CWE information
df['has_cwe'] = df['cwe'].notna() & (df['cwe'] != '')
df['cwe_primary'] = df['cwe'].str.extract(r'(CWE-\d+)')[0]  # Extract first CWE

print(f"CVEs with CWE: {df['has_cwe'].sum():,} ({df['has_cwe'].mean()*100:.1f}%)")
print(f"Unique CWEs: {df['cwe_primary'].nunique():,}")
print(f"\nTop 10 CWEs:")
print(df['cwe_primary'].value_counts().head(10))

In [ ]:
# Temporal split for validation
split_date = pd.Timestamp('2024-11-01')

train_df = df[df['published'] < split_date].copy()
test_df = df[df['published'] >= split_date].copy()

print(f"Training set: {len(train_df):,} CVEs (before {split_date.date()})")
print(f"Test set: {len(test_df):,} CVEs (from {split_date.date()})")
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts(normalize=True).round(3))
print(f"\nTest label distribution:")
print(test_df['label'].value_counts(normalize=True).round(3))

## 2. Baseline: LambdaRank Model

Load and evaluate the existing LambdaRank model as our baseline.

In [ ]:
# Load pre-trained model
import pickle

model_path = Path('../models/ltr_model_conf_weighted.pkl')
with open(model_path, 'rb') as f:
    baseline_model = pickle.load(f)

print(f"✓ Loaded baseline model: {model_path.name}")
print(f"Model type: {type(baseline_model).__name__}")

In [ ]:
# Feature engineering for baseline
feature_cols = [
    'cvss_norm', 'epss_score', 'epss_percentile', 'kev_flag',
    'recency_score', 'attack_technique_count', 'has_attack',
    'chpl_flag', 'is_healthcare', 'cvss_epss_product',
    'kev_healthcare_interaction'
]

train_features = create_all_features(train_df, feature_cols)
test_features = create_all_features(test_df, feature_cols)

X_train = train_features[feature_cols]
y_train = train_features['label']
X_test = test_features[feature_cols]
y_test = test_features['label']

print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")

In [ ]:
# Baseline predictions
baseline_scores = baseline_model.predict(X_test)
test_df['baseline_score'] = baseline_scores

# Evaluate baseline
from src.evaluation.metrics import ndcg_at_k, precision_at_k

k_values = [10, 20, 50, 100]
baseline_results = {}

for k in k_values:
    ndcg = ndcg_at_k(y_test, baseline_scores, k)
    prec = precision_at_k(y_test, baseline_scores, k, threshold=3)
    baseline_results[f'NDCG@{k}'] = ndcg
    baseline_results[f'Precision@{k}'] = prec

print("Baseline LambdaRank Performance:")
print("=" * 40)
for metric, value in baseline_results.items():
    print(f"{metric:20s}: {value:.4f}")

# Healthcare-specific metrics
healthcare_test = test_df[test_df['is_healthcare'] == 1]
if len(healthcare_test) > 0:
    top_100_healthcare = test_df.nlargest(100, 'baseline_score')['is_healthcare'].sum()
    print(f"\nHealthcare CVEs in Top-100: {top_100_healthcare}")
    print(f"Healthcare Recall@100: {top_100_healthcare / len(healthcare_test):.2%}")

## 3. Graph Construction

Build two types of graphs:
1. **CVE-CWE Bipartite Graph**: Connects CVEs to their weakness types
2. **CVE Similarity Graph**: Connects similar CVEs based on features

In [ ]:
# 3.1: CVE-CWE Bipartite Graph
G_bipartite = nx.Graph()

# Add CVE nodes
for idx, row in train_df.iterrows():
    G_bipartite.add_node(
        row['cve_id'],
        node_type='cve',
        cvss=row['cvss'],
        epss=row['epss_score'],
        kev=row['kev_flag'],
        label=row['label']
    )

# Add CWE nodes and edges
cve_cwe_edges = 0
for idx, row in train_df[train_df['has_cwe']].iterrows():
    cwe = row['cwe_primary']
    if cwe:
        if cwe not in G_bipartite:
            G_bipartite.add_node(cwe, node_type='cwe')
        G_bipartite.add_edge(row['cve_id'], cwe, edge_type='has_weakness')
        cve_cwe_edges += 1

print(f"Bipartite Graph:")
print(f"  CVE nodes: {sum(1 for n, d in G_bipartite.nodes(data=True) if d.get('node_type') == 'cve'):,}")
print(f"  CWE nodes: {sum(1 for n, d in G_bipartite.nodes(data=True) if d.get('node_type') == 'cwe'):,}")
print(f"  Edges: {G_bipartite.number_of_edges():,}")
print(f"  Density: {nx.density(G_bipartite):.6f}")

In [ ]:
# 3.2: CVE Similarity Graph
# Create feature vectors for similarity
similarity_features = ['cvss', 'epss_score', 'kev_flag', 'is_healthcare', 'attack_technique_count']
feature_matrix = train_df[similarity_features].fillna(0).values

# Normalize features
scaler = StandardScaler()
feature_matrix_normalized = scaler.fit_transform(feature_matrix)

# Compute cosine similarity
similarity_matrix = cosine_similarity(feature_matrix_normalized)

# Create similarity graph (keep only top-k similar CVEs per node)
k_neighbors = 10
similarity_threshold = 0.7

G_similarity = nx.Graph()
G_similarity.add_nodes_from(train_df['cve_id'])

edges_added = 0
for i, cve_i in enumerate(train_df['cve_id']):
    # Get top-k most similar CVEs
    similarities = similarity_matrix[i]
    top_k_indices = np.argsort(similarities)[::-1][1:k_neighbors+1]  # Exclude self
    
    for j in top_k_indices:
        if similarities[j] >= similarity_threshold:
            cve_j = train_df.iloc[j]['cve_id']
            G_similarity.add_edge(cve_i, cve_j, weight=similarities[j])
            edges_added += 1

print(f"\nSimilarity Graph:")
print(f"  Nodes: {G_similarity.number_of_nodes():,}")
print(f"  Edges: {G_similarity.number_of_edges():,}")
print(f"  Avg degree: {sum(dict(G_similarity.degree()).values()) / G_similarity.number_of_nodes():.2f}")
print(f"  Connected components: {nx.number_connected_components(G_similarity):,}")

## 4. DiffusionRank Algorithm

Random walk with restart on the similarity graph to propagate priority scores.

In [ ]:
def diffusion_rank(G, seed_scores, alpha=0.85, max_iter=100, tol=1e-6):
    """
    DiffusionRank: Random walk with restart for priority score propagation.
    
    Args:
        G: NetworkX graph
        seed_scores: Dict of {node: initial_score}
        alpha: Restart probability (higher = more influence from seeds)
        max_iter: Maximum iterations
        tol: Convergence tolerance
    
    Returns:
        Dict of {node: diffusion_score}
    """
    nodes = list(G.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    # Initialize seed vector
    seed_vector = np.zeros(n)
    for node, score in seed_scores.items():
        if node in node_to_idx:
            seed_vector[node_to_idx[node]] = score
    
    # Normalize seed vector
    if seed_vector.sum() > 0:
        seed_vector = seed_vector / seed_vector.sum()
    
    # Build transition matrix
    adj_matrix = nx.to_numpy_array(G, nodelist=nodes, weight='weight')
    row_sums = adj_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # Avoid division by zero
    transition_matrix = adj_matrix / row_sums
    
    # Initialize rank vector
    rank_vector = seed_vector.copy()
    
    # Iterative propagation
    for iteration in range(max_iter):
        rank_vector_new = (1 - alpha) * transition_matrix.T @ rank_vector + alpha * seed_vector
        
        # Check convergence
        diff = np.abs(rank_vector_new - rank_vector).sum()
        if diff < tol:
            print(f"  Converged after {iteration+1} iterations")
            break
        
        rank_vector = rank_vector_new
    
    # Convert back to dict
    scores = {nodes[i]: rank_vector[i] for i in range(n)}
    return scores

In [ ]:
# Create seed scores from baseline model
train_scores = baseline_model.predict(X_train)
seed_scores = dict(zip(train_df['cve_id'], train_scores))

print("Running DiffusionRank...")
diffusion_scores = diffusion_rank(G_similarity, seed_scores, alpha=0.85)

# Add to dataframe
train_df['diffusion_score'] = train_df['cve_id'].map(diffusion_scores)

print(f"\nDiffusion scores computed for {len(diffusion_scores):,} CVEs")
print(f"Score range: [{min(diffusion_scores.values()):.6f}, {max(diffusion_scores.values()):.6f}]")

# For test set, use baseline scores (no graph available)
test_df['diffusion_score'] = test_df['baseline_score']  # Fallback

In [ ]:
# Visualize score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['baseline_score'], bins=50, alpha=0.7, label='Baseline')
axes[0].hist(train_df['diffusion_score'], bins=50, alpha=0.7, label='DiffusionRank')
axes[0].set_xlabel('Priority Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Score Distribution Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(train_df['baseline_score'], train_df['diffusion_score'], alpha=0.3, s=10)
axes[1].plot([0, 1], [0, 1], 'r--', label='y=x')
axes[1].set_xlabel('Baseline Score')
axes[1].set_ylabel('DiffusionRank Score')
axes[1].set_title('Score Correlation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

correlation = train_df[['baseline_score', 'diffusion_score']].corr().iloc[0, 1]
print(f"\nCorrelation: {correlation:.4f}")

## 5. Model Comparison

Compare baseline LambdaRank vs DiffusionRank performance.

In [ ]:
# Evaluate DiffusionRank on test set
diffusion_results = {}

for k in k_values:
    ndcg = ndcg_at_k(y_test, test_df['diffusion_score'], k)
    prec = precision_at_k(y_test, test_df['diffusion_score'], k, threshold=3)
    diffusion_results[f'NDCG@{k}'] = ndcg
    diffusion_results[f'Precision@{k}'] = prec

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Baseline': baseline_results,
    'DiffusionRank': diffusion_results
})
comparison_df['Improvement'] = ((comparison_df['DiffusionRank'] - comparison_df['Baseline']) / comparison_df['Baseline'] * 100).round(2)
comparison_df['Improvement'] = comparison_df['Improvement'].astype(str) + '%'

print("\nModel Comparison:")
print("=" * 60)
print(comparison_df.to_string())

# Highlight best model per metric
print("\n" + "=" * 60)
for metric in comparison_df.index:
    best_model = comparison_df.loc[metric, ['Baseline', 'DiffusionRank']].idxmax()
    best_value = comparison_df.loc[metric, best_model]
    print(f"{metric:20s}: {best_model:15s} ({best_value:.4f})")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

metrics = [m for m in comparison_df.index if 'NDCG' in m]
x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, [baseline_results[m] for m in metrics], width, label='Baseline', alpha=0.8)
ax.bar(x + width/2, [diffusion_results[m] for m in metrics], width, label='DiffusionRank', alpha=0.8)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('NDCG Comparison: Baseline vs DiffusionRank', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0.85, 1.0])

plt.tight_layout()
plt.show()

## 6. Top-K Analysis

Examine the highest priority CVEs identified by each model.

In [ ]:
# Top-20 CVEs from each model
top_20_baseline = test_df.nlargest(20, 'baseline_score')[[
    'cve_id', 'cvss', 'epss_score', 'kev_flag', 'is_healthcare', 'label', 'baseline_score'
]]

top_20_diffusion = test_df.nlargest(20, 'diffusion_score')[[
    'cve_id', 'cvss', 'epss_score', 'kev_flag', 'is_healthcare', 'label', 'diffusion_score'
]]

print("Top-20 CVEs: Baseline LambdaRank")
print("=" * 80)
print(top_20_baseline.to_string(index=False))

print("\n\nTop-20 CVEs: DiffusionRank")
print("=" * 80)
print(top_20_diffusion.to_string(index=False))

In [ ]:
# Overlap analysis
baseline_top_ids = set(top_20_baseline['cve_id'])
diffusion_top_ids = set(top_20_diffusion['cve_id'])

overlap = baseline_top_ids & diffusion_top_ids
baseline_only = baseline_top_ids - diffusion_top_ids
diffusion_only = diffusion_top_ids - baseline_top_ids

print(f"\nTop-20 Overlap Analysis:")
print(f"  Overlap: {len(overlap)} CVEs")
print(f"  Baseline only: {len(baseline_only)} CVEs")
print(f"  DiffusionRank only: {len(diffusion_only)} CVEs")

if len(diffusion_only) > 0:
    print(f"\nCVEs uniquely identified by DiffusionRank:")
    unique_diffusion = test_df[test_df['cve_id'].isin(diffusion_only)][[
        'cve_id', 'cvss', 'kev_flag', 'is_healthcare', 'label', 'cwe_primary'
    ]]
    print(unique_diffusion.to_string(index=False))

## 7. Summary & Conclusions

Key findings from advanced model experiments.

In [ ]:
# Summary statistics
print("=" * 80)
print("PHASE 7 ADVANCED MODELS - SUMMARY")
print("=" * 80)

print(f"\n1. DATA STATISTICS")
print(f"   Training CVEs: {len(train_df):,}")
print(f"   Test CVEs: {len(test_df):,}")
print(f"   CVEs with CWE: {train_df['has_cwe'].sum():,}")
print(f"   Unique CWEs: {train_df['cwe_primary'].nunique():,}")

print(f"\n2. GRAPH STATISTICS")
print(f"   Bipartite graph edges: {G_bipartite.number_of_edges():,}")
print(f"   Similarity graph edges: {G_similarity.number_of_edges():,}")
print(f"   Avg CVE similarity degree: {sum(dict(G_similarity.degree()).values()) / G_similarity.number_of_nodes():.2f}")

print(f"\n3. PERFORMANCE COMPARISON")
print(f"   Baseline NDCG@20: {baseline_results['NDCG@20']:.4f}")
print(f"   DiffusionRank NDCG@20: {diffusion_results['NDCG@20']:.4f}")
improvement = (diffusion_results['NDCG@20'] - baseline_results['NDCG@20']) / baseline_results['NDCG@20'] * 100
print(f"   Improvement: {improvement:+.2f}%")

print(f"\n4. NEXT STEPS")
print(f"   - Implement RGCN model with PyTorch Geometric")
print(f"   - Create ensemble combining all models")
print(f"   - Add GPU acceleration for training")
print(f"   - Expand graph with CVE-Product relationships")

print("\n" + "=" * 80)
print(f"Notebook completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

In [ ]:
# Evaluate RGCN model
rgcn_results = {}

for k in k_values:
    ndcg = ndcg_at_k(y_test, test_df['rgcn_score'], k)
    prec = precision_at_k(y_test, test_df['rgcn_score'], k, threshold=3)
    rgcn_results[f'NDCG@{k}'] = ndcg
    rgcn_results[f'Precision@{k}'] = prec

print("RGCN Model Performance:")
print("=" * 60)
for metric, value in rgcn_results.items():
    print(f"{metric:20s}: {value:.4f}")

# Compare with baseline
print("\n" + "=" * 60)
print("Improvement over Baseline:")
for metric in [f'NDCG@{k}' for k in k_values]:
    baseline_val = baseline_results[metric]
    rgcn_val = rgcn_results[metric]
    improvement = (rgcn_val - baseline_val) / baseline_val * 100
    print(f"{metric:20s}: {improvement:+.2f}%")

In [ ]:
# Generate RGCN predictions on test set
# First prepare test data
test_features = test_df[feature_cols].values
test_labels = test_df['label'].values

# Combine train and test for graph (RGCN needs full graph context)
all_features = np.vstack([train_features, test_features])
all_labels = np.concatenate([train_labels, test_labels])

# Update CVE-CWE mapping with test data
all_df = pd.concat([train_df, test_df], ignore_index=True)
full_cve_to_cwe_map = {}

for idx, row in all_df.iterrows():
    if pd.notna(row['cwe_id']):
        same_cwe_cves = all_df[all_df['cwe_id'] == row['cwe_id']].index.tolist()
        full_cve_to_cwe_map[idx] = [i for i in same_cwe_cves if i != idx][:10]
    else:
        full_cve_to_cwe_map[idx] = []

# Prepare full graph data
x, edge_index, edge_type, y, _, _, test_mask_full = prepare_rgcn_data(
    cve_features=all_features,
    cve_to_cwe=full_cve_to_cwe_map,
    cve_labels=all_labels,
    train_idx=np.arange(len(train_df)),
    test_idx=np.arange(len(train_df), len(all_df))
)

# Generate predictions
rgcn_predictions = rgcn_trainer.predict(x, edge_index, edge_type, test_mask_full)
test_df['rgcn_score'] = rgcn_predictions

print(f"✓ RGCN predictions generated for {len(rgcn_predictions):,} test CVEs")
print(f"Score range: [{rgcn_predictions.min():.4f}, {rgcn_predictions.max():.4f}]")

In [ ]:
# Visualize RGCN training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
epochs_range = range(1, len(rgcn_history['train_loss']) + 1)
axes[0].plot(epochs_range, rgcn_history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, rgcn_history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('RGCN Training Curves', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Loss improvement
train_loss_improvement = (rgcn_history['train_loss'][0] - rgcn_history['train_loss'][-1]) / rgcn_history['train_loss'][0] * 100
val_loss_improvement = (rgcn_history['val_loss'][0] - rgcn_history['val_loss'][-1]) / rgcn_history['val_loss'][0] * 100

improvements = [train_loss_improvement, val_loss_improvement]
labels = ['Train', 'Validation']
colors = ['steelblue', 'darkorange']

axes[1].bar(labels, improvements, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Loss Reduction (%)', fontsize=12)
axes[1].set_title('RGCN Loss Improvement', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for i, v in enumerate(improvements):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Train RGCN model
print("Training RGCN model...")
print("This may take 2-3 minutes on CPU, faster on GPU/MPS\n")

rgcn_model, rgcn_trainer, rgcn_history = train_rgcn_model(
    cve_features=train_features,
    cve_to_cwe=cve_to_cwe_map,
    cve_labels=train_labels,
    train_idx=rgcn_train_idx,
    val_idx=rgcn_val_idx,
    hidden_channels=64,
    num_layers=2,
    dropout=0.3,
    learning_rate=0.01,
    epochs=100,
    early_stopping_patience=15,
    device=None,  # Auto-detect GPU/MPS/CPU
    verbose=True
)

print(f"\n✓ RGCN training complete!")
print(f"Final train loss: {rgcn_history['train_loss'][-1]:.4f}")
print(f"Final val loss: {rgcn_history['val_loss'][-1]:.4f}")
print(f"Total epochs: {len(rgcn_history['train_loss'])}")

In [ ]:
# Prepare features and labels for RGCN
train_features = train_df[feature_cols].values
train_labels = train_df['label'].values

# Create train/val split indices
train_size = int(0.8 * len(train_df))
rgcn_train_idx = np.arange(train_size)
rgcn_val_idx = np.arange(train_size, len(train_df))

print(f"RGCN data prepared:")
print(f"  Features shape: {train_features.shape}")
print(f"  Train samples: {len(rgcn_train_idx):,}")
print(f"  Val samples: {len(rgcn_val_idx):,}")

In [ ]:
# Prepare CVE-CWE mapping for RGCN
# Map CVE indices to connected CWE indices
cve_to_cwe_map = {}

for idx, row in train_df.iterrows():
    # Find other CVEs with same CWE
    if pd.notna(row['cwe_id']):
        same_cwe_cves = train_df[train_df['cwe_id'] == row['cwe_id']].index.tolist()
        # Map this CVE to other CVEs in same CWE (as "connected through CWE")
        cve_to_cwe_map[idx] = [i for i in same_cwe_cves if i != idx][:10]  # Limit to 10 connections
    else:
        cve_to_cwe_map[idx] = []

print(f"CVE-CWE mapping created")
print(f"Total CVEs: {len(cve_to_cwe_map):,}")
print(f"CVEs with connections: {sum(1 for v in cve_to_cwe_map.values() if len(v) > 0):,}")
print(f"Average connections per CVE: {np.mean([len(v) for v in cve_to_cwe_map.values()]):.2f}")

In [ ]:
# Import RGCN module
import torch
from src.models.rgcn import (
    RGCNPrioritizer,
    CVERGCNTrainer,
    train_rgcn_model,
    prepare_rgcn_data
)

print(f"✓ RGCN module imported")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 7.5. RGCN (Relational Graph Convolutional Network)

Train a deep learning model that learns CVE embeddings from the graph structure.

## 8. Ensemble Methods

Combine baseline and DiffusionRank predictions using various ensemble strategies.

In [ ]:
# Import ensemble module
from src.models.ensemble import (
    EnsembleRanker,
    bootstrap_ensemble,
    evaluate_ensemble_diversity
)

print("✓ Ensemble module imported")

In [ ]:
# Prepare predictions for ensemble
train_predictions = {
    'baseline': train_df['baseline_score'].values,
    'diffusion': train_df['diffusion_score'].values
}

test_predictions = {
    'baseline': test_df['baseline_score'].values,
    'diffusion': test_df['diffusion_score'].values
}

print(f"Train predictions: {len(train_predictions['baseline']):,} samples")
print(f"Test predictions: {len(test_predictions['baseline']):,} samples")

In [ ]:
# Evaluate model diversity
diversity_metrics = evaluate_ensemble_diversity(test_predictions, k=100)

print("Model Diversity Analysis:")
print("=" * 50)
for metric, value in diversity_metrics.items():
    print(f"{metric:30s}: {value:.4f}")

print(f"\nInterpretation:")
if diversity_metrics['diversity_score'] > 0.3:
    print("✓ Good diversity - ensemble will likely improve performance")
elif diversity_metrics['diversity_score'] > 0.1:
    print("⚠ Moderate diversity - ensemble may provide small gains")
else:
    print("✗ Low diversity - models are very similar")

In [ ]:
# 1. Simple Average Ensemble
ensemble_simple = EnsembleRanker(method='simple_average')
ensemble_simple.fit(train_predictions, train_df['label'].values)
simple_scores = ensemble_simple.predict(test_predictions)
test_df['ensemble_simple'] = simple_scores

print("1. Simple Average Ensemble")
print("   Weights:", ensemble_simple.get_weights())

In [ ]:
# 2. Weighted Average Ensemble (learns optimal weights)
ensemble_weighted = EnsembleRanker(method='weighted_average')
ensemble_weighted.fit(train_predictions, train_df['label'].values)
weighted_scores = ensemble_weighted.predict(test_predictions)
test_df['ensemble_weighted'] = weighted_scores

print("\n2. Weighted Average Ensemble")
print("   Weights:", ensemble_weighted.get_weights())

In [ ]:
# 3. Meta-Learning Ensemble (Ridge regression)
ensemble_meta = EnsembleRanker(method='meta_learning', meta_model='ridge')
ensemble_meta.fit(train_predictions, train_df['label'].values)
meta_scores = ensemble_meta.predict(test_predictions)
test_df['ensemble_meta'] = meta_scores

print("\n3. Meta-Learning Ensemble (Ridge)")
print("   Feature importance:", ensemble_meta.get_feature_importance())

In [ ]:
# 4. Rank Fusion Ensemble
ensemble_rank = EnsembleRanker(method='rank_fusion')
rank_scores = ensemble_rank._rank_fusion(test_predictions)
test_df['ensemble_rank'] = rank_scores

print("\n4. Reciprocal Rank Fusion")
print("   Combined rankings from both models")

In [ ]:
# Evaluate all ensemble methods
ensemble_results = {}

ensemble_methods = {
    'Simple Avg': 'ensemble_simple',
    'Weighted Avg': 'ensemble_weighted',
    'Meta-Learning': 'ensemble_meta',
    'Rank Fusion': 'ensemble_rank'
}

for method_name, score_col in ensemble_methods.items():
    method_results = {}
    for k in k_values:
        ndcg = ndcg_at_k(y_test, test_df[score_col], k)
        prec = precision_at_k(y_test, test_df[score_col], k, threshold=3)
        method_results[f'NDCG@{k}'] = ndcg
        method_results[f'Precision@{k}'] = prec
    ensemble_results[method_name] = method_results

# Add baseline and diffusion for comparison
ensemble_results['Baseline'] = baseline_results
ensemble_results['DiffusionRank'] = diffusion_results

# Create comparison DataFrame
ensemble_comparison = pd.DataFrame(ensemble_results).T

print("\nEnsemble Methods Comparison:")
print("=" * 100)
print(ensemble_comparison.round(4).to_string())

# Find best method per metric
print("\n" + "=" * 100)
print("Best Method Per Metric:")
for metric in ensemble_comparison.columns:
    best_method = ensemble_comparison[metric].idxmax()
    best_value = ensemble_comparison.loc[best_method, metric]
    print(f"{metric:20s}: {best_method:20s} ({best_value:.4f})")

In [ ]:
# Visualize ensemble performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# NDCG comparison
ndcg_metrics = [m for m in ensemble_comparison.columns if 'NDCG' in m]
x = np.arange(len(ndcg_metrics))
width = 0.12

for i, method in enumerate(['Baseline', 'DiffusionRank', 'Weighted Avg', 'Meta-Learning']):
    values = [ensemble_comparison.loc[method, m] for m in ndcg_metrics]
    axes[0].bar(x + i*width, values, width, label=method, alpha=0.8)

axes[0].set_xlabel('Metric', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('NDCG Comparison Across Methods', fontsize=14, fontweight='bold')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(ndcg_metrics, rotation=0)
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0.85, 1.0])

# Improvement over baseline
baseline_ndcg20 = baseline_results['NDCG@20']
improvements = []
methods_list = ['DiffusionRank', 'Simple Avg', 'Weighted Avg', 'Meta-Learning', 'Rank Fusion']

for method in methods_list:
    method_ndcg20 = ensemble_comparison.loc[method, 'NDCG@20']
    improvement = (method_ndcg20 - baseline_ndcg20) / baseline_ndcg20 * 100
    improvements.append(improvement)

colors = ['green' if x > 0 else 'red' for x in improvements]
axes[1].barh(methods_list, improvements, color=colors, alpha=0.7)
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Improvement over Baseline (%)', fontsize=12)
axes[1].set_title('NDCG@20 Improvement', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(improvements):
    axes[1].text(v + 0.05 if v > 0 else v - 0.15, i, f'{v:+.2f}%', 
                 va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Bootstrap Ensemble with Uncertainty

Quantify prediction uncertainty using bootstrap sampling.

In [ ]:
# Bootstrap ensemble for uncertainty quantification
print("Running bootstrap ensemble (100 iterations)...")
print("This may take a minute...")

mean_scores, std_scores = bootstrap_ensemble(
    train_predictions,
    train_df['label'].values,
    n_bootstrap=100,
    sample_size=0.8
)

# Map to test set predictions
train_df['bootstrap_mean'] = mean_scores
train_df['bootstrap_std'] = std_scores

print(f"\n✓ Bootstrap complete")
print(f"Mean score range: [{mean_scores.min():.4f}, {mean_scores.max():.4f}]")
print(f"Average uncertainty (std): {std_scores.mean():.4f}")
print(f"Max uncertainty: {std_scores.max():.4f}")

In [ ]:
# Analyze high uncertainty CVEs
uncertainty_threshold = std_scores.mean() + std_scores.std()
high_uncertainty_idx = std_scores > uncertainty_threshold
high_uncertainty_cves = train_df[high_uncertainty_idx][
    ['cve_id', 'cvss', 'epss_score', 'kev_flag', 'label', 'bootstrap_mean', 'bootstrap_std']
].sort_values('bootstrap_std', ascending=False)

print(f"\nHigh Uncertainty CVEs (>{uncertainty_threshold:.4f} std):")
print(f"Count: {len(high_uncertainty_cves):,}")
print("\nTop 10 most uncertain CVEs:")
print(high_uncertainty_cves.head(10).to_string(index=False))

# Characteristics of uncertain CVEs
print(f"\nCharacteristics of uncertain CVEs:")
print(f"  Average CVSS: {high_uncertainty_cves['cvss'].mean():.2f}")
print(f"  Average EPSS: {high_uncertainty_cves['epss_score'].mean():.4f}")
print(f"  KEV rate: {high_uncertainty_cves['kev_flag'].mean():.1%}")
print(f"  Average label: {high_uncertainty_cves['label'].mean():.2f}")

In [ ]:
# Visualize uncertainty
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uncertainty distribution
axes[0].hist(std_scores, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(std_scores.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0].axvline(uncertainty_threshold, color='orange', linestyle='--', linewidth=2, label='High uncertainty threshold')
axes[0].set_xlabel('Prediction Uncertainty (Std Dev)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Prediction Uncertainty', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Uncertainty vs prediction
axes[1].scatter(mean_scores, std_scores, alpha=0.3, s=20, c=train_df['label'], cmap='RdYlGn', vmin=1, vmax=5)
axes[1].set_xlabel('Mean Prediction Score', fontsize=12)
axes[1].set_ylabel('Uncertainty (Std Dev)', fontsize=12)
axes[1].set_title('Prediction Score vs Uncertainty', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar.set_label('True Label', fontsize=11)

plt.tight_layout()
plt.show()

## 10. Final Summary & Recommendations

Key findings and recommendations for production deployment.

In [ ]:
# Final performance summary
print("=" * 100)
print("PHASE 7 ADVANCED MODELS - FINAL SUMMARY")
print("=" * 100)

# Best performing model
best_model = ensemble_comparison['NDCG@20'].idxmax()
best_ndcg = ensemble_comparison.loc[best_model, 'NDCG@20']

print(f"\n1. BEST PERFORMING MODEL")
print(f"   Model: {best_model}")
print(f"   NDCG@20: {best_ndcg:.4f}")
print(f"   Improvement over baseline: {(best_ndcg - baseline_results['NDCG@20']) / baseline_results['NDCG@20'] * 100:+.2f}%")

print(f"\n2. MODEL RANKINGS (by NDCG@20)")
rankings = ensemble_comparison['NDCG@20'].sort_values(ascending=False)
for i, (model, score) in enumerate(rankings.items(), 1):
    print(f"   {i}. {model:20s}: {score:.4f}")

print(f"\n3. ENSEMBLE METHOD ANALYSIS")
weights = ensemble_weighted.get_weights()
print(f"   Optimal weights (Weighted Avg):")
for model, weight in weights.items():
    print(f"     - {model}: {weight:.3f}")

feature_imp = ensemble_meta.get_feature_importance()
print(f"\n   Meta-model importance:")
for model, imp in feature_imp.items():
    print(f"     - {model}: {imp:.3f}")

print(f"\n4. DIVERSITY METRICS")
for metric, value in diversity_metrics.items():
    print(f"   {metric:30s}: {value:.4f}")

print(f"\n5. UNCERTAINTY QUANTIFICATION")
print(f"   Average uncertainty: {std_scores.mean():.4f}")
print(f"   High uncertainty CVEs: {high_uncertainty_idx.sum():,} ({high_uncertainty_idx.mean()*100:.1f}%)")
print(f"   Uncertainty range: [{std_scores.min():.4f}, {std_scores.max():.4f}]")

print(f"\n6. HEALTHCARE-SPECIFIC PERFORMANCE")
if len(test_df[test_df['is_healthcare'] == 1]) > 0:
    healthcare_test = test_df[test_df['is_healthcare'] == 1]
    for method_name, score_col in [('Baseline', 'baseline_score'), ('Best Ensemble', ensemble_methods['Weighted Avg'])]:
        top_100 = test_df.nlargest(100, score_col)['is_healthcare'].sum()
        recall = top_100 / len(healthcare_test)
        print(f"   {method_name:20s}: {top_100} CVEs in Top-100 (Recall: {recall:.2%})")

print(f"\n7. RECOMMENDATIONS FOR PRODUCTION")
print(f"   ✓ Use {best_model} for deployment")
print(f"   ✓ Ensemble methods provide {(best_ndcg - baseline_results['NDCG@20']) / baseline_results['NDCG@20'] * 100:.1f}% improvement")
print(f"   ✓ Monitor high-uncertainty predictions for manual review")
print(f"   ✓ Retrain models monthly with new CVE data")
print(f"   ✓ Consider adding RGCN for further improvements")

print(f"\n8. NEXT STEPS")
print(f"   - Implement RGCN model with PyTorch Geometric")
print(f"   - Add GPU acceleration for faster training")
print(f"   - Create production API endpoint for ensemble predictions")
print(f"   - Set up continuous monitoring and retraining pipeline")
print(f"   - Generate weekly priority reports for security team")

print("\n" + "=" * 100)
print(f"Analysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 100)